In [1]:
import re
from pathlib import Path

from bs4 import BeautifulSoup

## Process Raw

In [2]:
raw_dir = Path("data/raw")
out_dir = Path("data/text")
START_MARKER = "SECURITIES AND EXCHANGE COMMISSION"

for html_path in sorted(raw_dir.glob("*.html")):
    txt_path = out_dir / (html_path.stem + ".txt")

    html = html_path.read_text(encoding="utf-8", errors="ignore")
    break

In [3]:
soup = BeautifulSoup(html, "lxml")


In [4]:
for tag in soup(["script", "style", "noscript"]):
    tag.decompose()

In [ ]:
text = soup.get_text(separator="\n")
#Keep only text after the first occurrence of the marker (case-insensitive)
upper_text = text.upper()
marker_upper = START_MARKER.upper()
idx = upper_text.find(marker_upper)
if idx != -1:
    text = text[idx + len(START_MARKER) :]

text = text.replace("\xa0", " ")

# Collapse multiple spaces
#text = re.sub(r" +", " ", text)
# Remove space before punctuation so "word ," and "word ." become "word," and "word."
#text = re.sub(r" ([,.;:!?)\]}\"])", r"\1", text)

# Normalise: strip lines and drop empties, then rejoin
lines = [line.strip() for line in text.splitlines()]
text = "\n".join(line for line in lines if line)
text = re.sub(r"\n([,.;:])", r"\1", text)
text = re.sub(r"\n{2,}", "\n", text)




In [16]:
print(text[:300])

Washington, D.C. 20549
FORM
10-K
(Mark One)
☒
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended
September 27, 2025
or
☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from


## Chunk + Embed

In [67]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv
import os
import json

In [ ]:

load_dotenv()  # loads variables from .env into environment
OPENAI_API_KEY = os.getenv("OPENAI_KEY")


In [31]:
PARTS = {
    "Part 0": ["FORM\n10-K(Mark One)"],
    "Part I": ["Item 1.", "Item 1A.","Item 1B.","Item 1C.","Item 2.","Item 3.","Item 4."],
    "Part II": ["Item 5.","Item 6.","Item 7.","Item 7A.","Item 8.","Item 9.","Item 9A.","Item 9B.","Item 9C."],
    "Part III": ["Item 10.","Item 11.","Item 12.","Item 13.","Item 14."],
    "Part IV": ["Item 15.","Item 16."]
}
SECTION_HEADERS = ["FORM\n10-K(Mark One)", "Item 1.", 
"Item 1A.",
"Item 1B.",
"Item 1C.",
"Item 2.",
"Item 3.",
"Item 4.",
"Item 5.",
"Item 6.",
"Item 7.",
"Item 7A.",
"Item 8.",
"Item 9.",
"Item 9A.",
"Item 9B.",
"Item 9C.",
"Item 10.",
"Item 11.",
"Item 12.",
"Item 13.",
"Item 14.",
"Item 15.",
"Item 16."]

In [ ]:
class ChunkEmbed:
    def __init__(self, folder_path: Path, key: str, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.folder_path = folder_path
        self.OPENAIkey = key
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.chunks_collection = []

    def init_embeddings(self): 
        embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

In [ ]:
def chunking(path: Path, chunks_col: list[dict], chunk_size: int = 1000, chunk_overlap: int = 200):
    """
    break text into sections based on the provided section headers.
    return a list of chunks, where each chunk is a dict with keys "chunk_id", "text", and "metadata".
    """
    #setting up the text splitter

    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,      # Max size of each chunk (measured by characters here)
    chunk_overlap=chunk_overlap    # Number of characters to overlap between adjacent chunks
    )
    text = path.read_text(encoding="utf-8")
    # get document id
    doc_id = text_path.stem
    # extract company name
    pattern = r'\n([^\n]*?)\s*\(Exact name of Registrant as specified in its charter\)'
    match = re.search(pattern,text,re.DOTALL)
    if match:
        company_name = match.group(1)
    else:
        company_name = "Unknown"
        print(f"Company name not found in {doc_id}")

    # first split into sections based on headers
    for i in range(1, len(SECTION_HEADERS)):
        header = SECTION_HEADERS[i-1]
        next_header = SECTION_HEADERS[i]
        start_idx = text.find(header)
        end_idx = text.find(next_header)
        # each section is the text between the current header and the next header
        section = text[start_idx:end_idx]
        # find part
        part = "Unknown"
        for p, sl in PARTS.items():
            if header in sl:
                part = p
                break
        # Split the text into chunks
        chunks = text_splitter.split_text(section)
        for j, c in enumerate(chunks):
            chunk_id = f"{doc_id}_section_{i+1}_chunk_{j+1}"
            metadata = {"company": company_name, "part":part, "section": header, "chunk": j+1}
            emb = embeddings.embed_query(c)
            chunk = {"chunk_id": chunk_id, "text": c, "embedding": emb, "metadata": metadata}
            chunks_col.append(chunk)

    return



In [35]:
Path("data/chunks/") / "chunks_collection.json"

WindowsPath('data/chunks/chunks_collection.json')

In [ ]:
PROCESSED_PATH=Path("data/text/")
OUTPUT_DIR = Path("data/chunks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks_collection = []
for text_path in PROCESSED_PATH.glob("*.txt"):
    chunking(path=text_path, chunks_col=chunks_collection, 
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    
    




In [50]:
for text_path in PROCESSED_PATH.glob("*.txt"):
    text = text_path.read_text(encoding="utf-8")
    pattern = r'\n([^\n]*?)\s*\(Exact name of'
    match = re.search(pattern,text,re.DOTALL)
    if match:
        company_name = match.group(1)
    else:
        company_name = "Unknown"
        print(f"Company name not found in {text_path.stem}")
        break

Company name not found in avgo-20251102


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import json

# langchain imports
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# loads variables from .env into environment
load_dotenv()  
OPENAI_API_KEY = os.getenv("OPENAI_KEY")

PROCESSED_PATH=Path("data/text/")
OUTPUT_DIR = Path("data/chunks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMB_MODEL = "text-embedding-3-small"

# relating sections to paths in the document. note: all document has same parts and sections
PARTS = {
    "Part 0": ["FORM\n10-K(Mark One)"],
    "Part I": ["Item 1.", "Item 1A.","Item 1B.","Item 1C.","Item 2.","Item 3.","Item 4."],
    "Part II": ["Item 5.","Item 6.","Item 7.","Item 7A.","Item 8.","Item 9.","Item 9A.","Item 9B.","Item 9C."],
    "Part III": ["Item 10.","Item 11.","Item 12.","Item 13.","Item 14."],
    "Part IV": ["Item 15.","Item 16."]
}

# section headers to look for when splitting the document into sections
SECTION_HEADERS = ["FORM\n10-K(Mark One)", "Item 1.", 
"Item 1A.",
"Item 1B.",
"Item 1C.",
"Item 2.",
"Item 3.",
"Item 4.",
"Item 5.",
"Item 6.",
"Item 7.",
"Item 7A.",
"Item 8.",
"Item 9.",
"Item 9A.",
"Item 9B.",
"Item 9C.",
"Item 10.",
"Item 11.",
"Item 12.",
"Item 13.",
"Item 14.",
"Item 15.",
"Item 16."]



class ChunkEmbed:
    def __init__(self, folder_path: Path, key: str, chunk_size: int = 1000, chunk_overlap: int = 200,
                 emb_model: str = "text-embedding-3-small"):
        self.folder_path = folder_path
        self.OPENAIkey = key
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.emb_model = emb_model
        self.chunks_collection = []
        self.embeddings = self.init_embeddings()
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=self.chunk_size, 
                                                            chunk_overlap=self.chunk_overlap)

    def init_embeddings(self) -> OpenAIEmbeddings: 
        """initialize the OpenAI embeddings model."""
        embeddings = OpenAIEmbeddings(model=self.emb_model,
            openai_api_key=self.OPENAIkey)
        return embeddings
    
    def chunk_all(self) -> None:
        """iterate through all text files in the folder and chunk them."""
        for text_path in self.folder_path.glob("*.txt"):
            print(f"Processing {text_path}...")
            self.chunking(text_path)


    def find_part(self, header: str) -> str:
        # find part
        part = "Unknown"
        for p, sl in PARTS.items():
            if header in sl:
                part = p
                break
        return part
    
    def chunking(self, path: Path) -> None:
        """
        break text into sections based on the provided section headers.
        return a list of chunks, where each chunk is a dict with keys "chunk_id", "text", and "metadata".
        """
        #setting up the text splitter

        text = path.read_text(encoding="utf-8")
        # get document id
        doc_id = path.stem
        stock_sym = path.stem.split("-")[0]

        # first split into sections based on headers
        idx = 0
        for i in range(1, len(SECTION_HEADERS)):
            header = SECTION_HEADERS[i-1]
            next_header = SECTION_HEADERS[i]
            start_idx = text.find(header)
            end_idx = text.find(next_header)
            # each section is the text between the current header and the next header
            section = text[start_idx:end_idx]
            # find part
            part = self.find_part(header)
            # Split the text into chunks
            chunks = self.text_splitter.split_text(section)
            for j, c in enumerate(chunks):
                chunk_id = f"{doc_id}_section_{i+1}_chunk_{j+1}"
                metadata = {"chunk_idx": idx, "source": path, "doc_id": doc_id,"emb_model": self.emb_model,
                    "stock symbol": stock_sym, "part":part, "section": header, "chunk": j+1}
                emb = self.embeddings.embed_query(c)
                chunk = {"chunk_id": chunk_id, "text": c, "embedding": emb, "metadata": metadata}
                self.chunks_collection.append(chunk)
                idx += 1

    def save_chunks(self, output_dir: Path):
        """save the chunks collection to a file."""
        output_path = OUTPUT_DIR / "chunks_collection.jsonl"
        #output_path.mkdir(parents=True, exist_ok=True)

        with output_path.open("w", encoding="utf-8") as f:
            for chunk in self.chunks_collection:
                json_line = json.dumps(chunk, ensure_ascii=False)
                f.write(json_line + "\n")
        print(f"Saved {len(self.chunks_collection)} chunks to {output_path}")








In [54]:
processor = ChunkEmbed(folder_path=PROCESSED_PATH, key=OPENAI_API_KEY, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
processor.chunk_all()
#processor.save_chunks(OUTPUT_DIR)

Processing data\text\aapl-20250927.txt...
Processing data\text\amd-20241228.txt...
Processing data\text\avgo-20251102.txt...
Processing data\text\ba-20241231.txt...
Processing data\text\brka-20241231.txt...
Processing data\text\cost-20250831.txt...
Processing data\text\crsp-20241231.txt...
Processing data\text\dpz-20241229.txt...
Processing data\text\goog-20251103.txt...
Processing data\text\ibm-20241231.txt...
Processing data\text\intc-20241228.txt...
Processing data\text\isrg-20241231.txt...
Processing data\text\meta-20241231.txt...
Processing data\text\mu-20250828.txt...
Processing data\text\nvda-20250126.txt...
Processing data\text\smr-20241231.txt...
Processing data\text\snps-20251031.txt...
Processing data\text\tsla-20241231.txt...
Processing data\text\unh-20241231.txt...


In [56]:
output_path = OUTPUT_DIR / "chunks_collection.jsonl"

In [57]:
output_path

WindowsPath('data/chunks/chunks_collection.jsonl')

In [73]:
with open('data/chunks/chunks_collection.jsonl', 'w', encoding="utf-8") as f:
    for c in processor.chunks_collection:
        c["metadata"]["source"] = str(c["metadata"]["source"])
        json_line = json.dumps(c, ensure_ascii=False)
        f.write(json_line + "\n")

In [74]:
idx= 3
for i in range(idx-1, -1, -1):
    print(i)

2
1
0
